In [10]:
import numpy as np
import pandas as pd

In [11]:
vws_dataset = pd.read_parquet(
    "../data/processed/vws_dataset.parquet"
)

In [12]:
wf_mean_ds = pd.read_parquet(
    "../data/processed/wf_mean_ds.parquet"
)

In [16]:
mean_weather_feature = pd.read_parquet(
    "../data/processed/mean_weather_feature.parquet"
)

In [17]:
vws_dataset = vws_dataset.drop(columns=["ws10"])
wf_mean_ds = wf_mean_ds.drop(columns=["ws10"])
mean_weather_feature = mean_weather_feature.drop(columns=["ws10"])

<h1>Modelling Datasets for evaluation models

In [18]:
def create_features(df):

    df = df.copy()

    # -----------------------------
    # Ensure datetime
    # -----------------------------
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # -----------------------------
    # Time Features
    # -----------------------------
    df["hour"] = df["timestamp"].dt.hour
    df["month"] = df["timestamp"].dt.month
    df["dayofyear"] = df["timestamp"].dt.dayofyear

    # -----------------------------
    # Cyclical Encoding
    # -----------------------------
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["dayofyear_sin"] = np.sin(
        2 * np.pi * df["dayofyear"] / 365
    )

    df["dayofyear_cos"] = np.cos(
        2 * np.pi * df["dayofyear"] / 365
    )

    # -----------------------------
    # Wind Direction Encoding
    # -----------------------------
    df["wd100_sin"] = np.sin(
        np.deg2rad(df["wd100"])
    )

    df["wd100_cos"] = np.cos(
        np.deg2rad(df["wd100"])
    )

    # -----------------------------
    # Meteorological Lag Features
    # -----------------------------
    lag_features = [
        "ws100",
        "wind_shear",
        "i10fg",
        "relative_humidity",
        "air_density",
        "blh",
        "tcc",
        "tp"
    ]

    for col in lag_features:

        df[f"{col}_lag_24"] = (
            df[col].shift(24)
        )

        df[f"{col}_lag_168"] = (
            df[col].shift(168)
        )

    # -----------------------------
    # Generation Lags
    # -----------------------------
    df["gen_lag_24"] = (
        df["WIND"].shift(24)
    )

    df["gen_lag_24"] = (
        df["WIND"].shift(24)
    )

    # -----------------------------
    # Rolling Features of wind generation, ws100 and i10fg
    # -----------------------------
    for col in ["WIND","ws100", "i10fg"]:

        df[f"{col}_roll_mean_24"] = (
        df[col]
        .shift(1)
        .rolling(24)
        .mean()
    )

        df[f"{col}_roll_std_24"] = (
        df[col]
        .shift(1)
        .rolling(24)
        .std()
    )

    # -----------------------------
    # Drop NaNs from lagging
    # -----------------------------
    df = df.dropna()

    return df

In [19]:
datasets = {
    "uk_mean": mean_weather_feature,
    "windfarm_mean": wf_mean_ds,
    "gvws": vws_dataset
}

processed_datasets = {}

for name, df in datasets.items():

    processed = create_features(df)

    processed_datasets[name] = processed

    print(
        f"{name}: {processed.shape}"
    )

uk_mean: (69960, 45)
windfarm_mean: (69960, 45)
gvws: (69960, 45)


In [20]:
for name, df in processed_datasets.items():

    filename = f"{name}_processed.parquet"

    df.to_parquet(
        f"../data/processed/{filename}",
        index=False
    )

    print(f"Saved {filename}")

Saved uk_mean_processed.parquet
Saved windfarm_mean_processed.parquet
Saved gvws_processed.parquet


In [21]:
%reset